# Одноимпульсные манёвры
## Примеры лекции 3 и домашнее задание 2

В первой части приведены решения всех численных примеров лекции: переходы между
заданными орбитами, коррекция $a,e$, коррекция $a,\omega$, коррекция $a,e,\Omega$
и выбор места манёвра. Во второй части находятся три домашних задания:
первые два обязательны, третье — по желанию (повышенной сложности).

Выполняйте ячейки сверху вниз. Все вспомогательные функции находятся в этом файле, остаётся собрать их как конструктор.

Расчёты выполняются в **километрах, секундах и радианах**. В таблицах углы
показаны в градусах, а импульсы — в м/с. Порядок элементов: $a,e,\omega,i,\Omega$.
Истинные аномалии приводим к $0\le\nu<360^\circ$.


In [ ]:
from dataclasses import dataclass, replace
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Circle
from IPython.display import display, HTML
from html import escape

# Во всём ноутбуке: км, с, радианы. Только выводимая Δv — в м/с.
MU = 398600.44158
R_EARTH = 6378.137
DEG = np.pi / 180.0
TAU = 2.0 * np.pi
COLORS = dict(paper='#FBF9F3', ink='#302B28', blue='#187AA5',
              muted='#6D6560', rule='#A9CADA', tint='#EDF4F6', teal='#397B73')
FONT = 'PT Serif' if any(f.name == 'PT Serif' for f in font_manager.fontManager.ttflist) else 'DejaVu Serif'
plt.rcParams.update({
    'font.family': [FONT, 'DejaVu Serif'], 'font.size': 11, 'mathtext.fontset': 'dejavuserif',
    'figure.figsize': (9.5, 4.8), 'figure.dpi': 120,
    'figure.facecolor': COLORS['paper'], 'axes.facecolor': COLORS['paper'],
    'axes.edgecolor': COLORS['muted'], 'axes.labelcolor': COLORS['ink'],
    'text.color': COLORS['ink'], 'xtick.color': COLORS['muted'],
    'ytick.color': COLORS['muted'], 'grid.color': COLORS['rule'],
    'grid.alpha': 0.65, 'grid.linewidth': 0.6, 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.titlesize': 13,
    'legend.frameon': False, 'lines.linewidth': 1.8,
})
BRANCH_COLORS = [COLORS['blue'], COLORS['muted'], COLORS['teal'], COLORS['ink']]
BRANCH_STYLES = ['-', '--', '-.', ':']
np.set_printoptions(precision=7, suppress=True)


## 1. Вспомогательные функции

`solve_acos_bsin_c_eq_0(A,B,C)` решает $A\cos x+B\sin x+C=0$.
Корни лежат в $[0,2\pi)$; два решения возвращаются в порядке
$\varphi-\alpha$, $\varphi+\alpha$.

`solve_quadratic(A,B,C)` решает $Ax^2+Bx+C=0$ и возвращает
различные действительные корни по убыванию. Линейный случай тоже обрабатывается.

Обе функции возвращают `[]`, если решений нет. Тождество $0=0$ вызывает
`InfiniteSolutionsError`: у него бесконечно много решений.


In [ ]:
class InfiniteSolutionsError(ValueError):
    """Уравнение тождественно: конечного списка корней не существует."""


def normalize_angle(angle):
    """Угол в радианах в диапазоне [0, 2π)."""
    if not np.isfinite(angle):
        raise ValueError('Угол должен быть конечным числом.')
    value = float(angle) % TAU
    return 0.0 if value == TAU else value


def normalize(vector):
    """Единичный вектор; нулевой вектор нормировать нельзя."""
    vector = np.asarray(vector, dtype=float)
    length = np.linalg.norm(vector)
    if not np.isfinite(length) or length == 0:
        raise ValueError('Нужен конечный ненулевой вектор.')
    return vector / length


def solve_acos_bsin_c_eq_0(a, b, c, tol=1e-12):
    """a cos(x) + b sin(x) + c = 0. Все различные корни в [0, 2π).

    [] — нет корней; один корень — касание; два — обычный случай.
    0 = 0 вызывает InfiniteSolutionsError. Коэффициенты масштабируются.
    Порядок корней: φ−α, φ+α, без сортировки по приведённому углу.
    """
    coef = np.asarray([a, b, c], dtype=float)
    if not np.isfinite(coef).all():
        raise ValueError('Коэффициенты должны быть конечными.')
    scale = np.max(np.abs(coef))
    if scale == 0:
        raise InfiniteSolutionsError('0 = 0: любой угол является решением.')
    a, b, c = coef / scale
    rho = np.hypot(a, b)
    if rho == 0 or abs(c) > rho * (1 + tol):
        return []
    phi = np.arctan2(b, a)
    alpha = np.arccos(np.clip(-c / rho, -1.0, 1.0))
    roots = [normalize_angle(phi - alpha), normalize_angle(phi + alpha)]
    distance = abs(np.arctan2(np.sin(roots[1]-roots[0]), np.cos(roots[1]-roots[0])))
    return roots[:1] if distance <= tol else roots


def solve_quadratic(a, b, c):
    """Все различные действительные корни a*x²+b*x+c=0, по убыванию.

    Учитывает линейное уравнение, двойной и нулевой корни, отсутствие
    корней и тождество 0=0. Малый, но ненулевой a не отбрасывается.
    """
    coef = np.asarray([a, b, c], dtype=np.longdouble)
    if not np.isfinite(coef).all():
        raise ValueError('Коэффициенты должны быть конечными.')
    scale = np.max(np.abs(coef))
    if scale == 0:
        raise InfiniteSolutionsError('0 = 0: любое число является решением.')
    a, b, c = coef / scale
    if a == 0:
        return [] if b == 0 else [float(-c / b)]
    discriminant = b*b - 4*a*c
    roundoff = 16*np.finfo(float).eps*(b*b + 4*abs(a*c))
    if discriminant < -roundoff:
        return []
    if abs(discriminant) <= roundoff:
        return [float(-b / (2*a))]
    root_d = np.sqrt(discriminant)
    q = -0.5*(b + np.copysign(root_d, b))
    roots = sorted([float(q/a), float(c/q)], reverse=True)
    if not np.isfinite(roots).all():
        raise OverflowError('Корень не помещается в числовой тип float.')
    return roots


def solve_true_anomaly(radius, a, e):
    """Все ν для данного r на некруговом эллипсе; A — отход от перицентра.

    Результат отсортирован в [0, 2π). В апсиде возвращается один корень.
    Круговая орбита требует отдельного соглашения о начале отсчёта ν.
    """
    if not np.isfinite([radius, a, e]).all() or radius <= 0 or a <= 0 or not 0 < e < 1:
        raise ValueError('Нужны r>0, a>0 и 0<e<1.')
    return sorted(solve_acos_bsin_c_eq_0(e, 0.0, 1-a*(1-e*e)/radius))


def directed_angle(first, second, normal):
    """Ориентированный угол вокруг normal, в диапазоне [0, 2π)."""
    first, second, normal = map(normalize, (first, second, normal))
    return normalize_angle(np.arctan2(np.dot(normal, np.cross(first, second)),
                                      np.dot(first, second)))


### Кеплеровы элементы, базис и векторы состояния

В исходной и конечной орбитах используем одну геоцентрическую инерциальную систему.
Орты $\mathbf P,\mathbf Q$ направлены в перицентр и на $90^\circ$ вперёд по движению:
$$\mathbf r=\frac{p}{1+e\cos\nu}(\cos\nu\,\mathbf P+\sin\nu\,\mathbf Q),$$
$$\mathbf v=\sqrt{\frac{\mu}{p}}
\left[-\sin\nu\,\mathbf P+(e+\cos\nu)\mathbf Q\right],\qquad p=a(1-e^2).$$
Компоненты в орбитальной плоскости:
$$v_r=\sqrt{\mu/p}\,e\sin\nu,\qquad v_t=\sqrt{\mu/p}(1+e\cos\nu).$$
В общей точке $\Delta\mathbf v=\mathbf v_2-\mathbf v_1$.
Вычитать компоненты скоростей, записанные в разных базисах, нельзя.

В `Maneuver` хранятся только точка приложения импульса `point`
(элементы конечной орбиты $a,e,\omega,i,\Omega,\nu$) и вектор `dv`.
`make_maneuver` проверяет совпадение радиус-векторов и вычисляет
$\Delta\mathbf v=\mathbf v_2-\mathbf v_1$.
Орбита не пересекает Землю, если $r_{\text{п},2}>R_\oplus$.


In [ ]:
@dataclass(frozen=True)
class Orbit:
    """Кеплеровы элементы точки: a,e,ω,i,Ω,ν; a — км, углы — радианы."""
    a: float
    e: float
    w: float = 0.0
    i: float = 0.0
    raan: float = 0.0
    nu: float = 0.0

    def __post_init__(self):
        if not np.isfinite([self.a, self.e, self.w, self.i, self.raan, self.nu]).all():
            raise ValueError('Все элементы должны быть конечными.')
        if self.a <= 0 or not 0 <= self.e < 1 or not 0 <= self.i <= np.pi:
            raise ValueError('Нужны a>0, 0≤e<1, 0≤i≤π.')

    @property
    def p(self):
        return self.a*(1-self.e)*(1+self.e)

    @property
    def rp(self):
        return self.a*(1-self.e)

    @property
    def ra(self):
        return self.a*(1+self.e)


def node_basis(i, raan):
    """Орты восходящего узла n, поперечного направления m и нормали h."""
    n = np.array([np.cos(raan), np.sin(raan), 0.0])
    h = np.array([np.sin(i)*np.sin(raan), -np.sin(i)*np.cos(raan), np.cos(i)])
    return n, np.cross(h, n), h


def orbital_basis(orbit):
    """P направлен в перицентр, Q на 90° вперёд в плоскости, h=P×Q."""
    n, m, h = node_basis(orbit.i, orbit.raan)
    P = np.cos(orbit.w)*n + np.sin(orbit.w)*m
    Q = -np.sin(orbit.w)*n + np.cos(orbit.w)*m
    return P, Q, h


def radius_at(orbit, nu):
    return orbit.p/(1 + orbit.e*np.cos(nu))


def state_at(orbit, nu=None, mu=MU):
    """Радиус-вектор [км] и скорость [км/с] в инерциальной системе."""
    nu = orbit.nu if nu is None else nu
    P, Q, _ = orbital_basis(orbit)
    r = radius_at(orbit, nu)*(np.cos(nu)*P + np.sin(nu)*Q)
    v = np.sqrt(mu/orbit.p)*(-np.sin(nu)*P + (orbit.e+np.cos(nu))*Q)
    return r, v


def radial_transverse_speed(orbit, nu):
    factor = np.sqrt(MU/orbit.p)
    return factor*orbit.e*np.sin(nu), factor*(1+orbit.e*np.cos(nu))


def argument_latitude(r, i, raan):
    """Аргумент широты именно в новой плоскости; сначала проверяем её."""
    n, m, h = node_basis(i, raan)
    if abs(np.dot(normalize(r), h)) > 1e-10:
        raise ValueError('Заданная плоскость не содержит радиус-вектор.')
    return normalize_angle(np.arctan2(np.dot(r, m), np.dot(r, n)))


def plane_with_raan(r, raan):
    """Наклонение (0,π) плоскости через r при данной ДВУ, либо None.

    Если точка на заданной линии узлов, плоскостей бесконечно много.
    Если возможна только экваториальная плоскость, её ДВУ не определена.
    """
    rx, ry, rz = normalize(r)
    A = rx*np.sin(raan) - ry*np.cos(raan)
    if np.hypot(A, rz) < 1e-13:
        raise InfiniteSolutionsError('Точка на линии узлов: наклонение свободно.')
    i = float(np.arctan2(-rz, A) % np.pi)
    return None if abs(np.sin(i)) < 1e-12 else i


@dataclass(frozen=True)
class Maneuver:
    point: Orbit  # Точка приложения импульса в элементах конечной орбиты.
    dv: np.ndarray  # Вектор импульса в инерциальной системе, км/с.


def make_maneuver(initial, nu1, target, nu2, position_tolerance_km=1e-6):
    """Проверяет общую точку и вычисляет импульс Δv = v₂ − v₁."""
    r1, v1 = state_at(initial, nu1)
    r2, v2 = state_at(target, nu2)
    error = float(np.linalg.norm(r2-r1))
    if error > position_tolerance_km:
        raise ValueError(f'Нет общей точки: невязка {error:.6g} км.')
    return Maneuver(replace(target, nu=normalize_angle(nu2)), v2-v1)


### Функции для примеров лекции

`correct_a_e`, `correct_a_w`, `correct_a_e_raan` возвращают список
из двух возможных манёвров, A и B. Если решения нет, на его месте стоит `None`.
При совпадении решений оба элемента списка содержат один и тот же манёвр.


In [ ]:
def correct_a_e(initial, nu1, a2, e2, plane=None):
    """A и B соответствуют двум значениям ν₂; None означает отсутствие решения.
    В апсиде оба значения ν₂ совпадают."""
    r1, _ = state_at(initial, nu1)
    roots = solve_true_anomaly(np.linalg.norm(r1), a2, e2)
    if not roots:
        return [None, None]
    i2, O2 = plane if plane is not None else (initial.i, initial.raan)
    u2 = argument_latitude(r1, i2, O2)
    if len(roots) == 1:
        roots = roots*2
    return [make_maneuver(initial, nu1,
                         Orbit(a2, e2, normalize_angle(u2-f), i2, O2), f)
            for f in roots]


def correct_a_w(initial, nu1, a2, w2):
    """A — больший корень e, B — меньший; порядок решений сохраняется."""
    radius = radius_at(initial, nu1)
    nu2 = normalize_angle(initial.w + nu1 - w2)
    roots = solve_quadratic(a2, radius*np.cos(nu2), radius-a2)
    if len(roots) == 1:
        roots = roots*2
    result = [None, None]
    for j, e2 in enumerate(roots):
        if 1e-10 < e2 < 1-1e-10:
            target = Orbit(a2, e2, w2, initial.i, initial.raan)
            result[j] = make_maneuver(initial, nu1, target, nu2)
    return result


def correct_a_e_raan(initial, nu1, a2, e2, raan2):
    r1, _ = state_at(initial, nu1)
    i2 = plane_with_raan(r1, raan2)
    if i2 is None:
        return [None, None]
    return correct_a_e(initial, nu1, a2, e2, (i2, normalize_angle(raan2)))


### Построение графиков и вывод результатов

Следующая ячейка содержит вспомогательный код для графиков и таблиц.
Его не нужно ни просматривать, ни понимать. Просто выполните ячейку.


In [ ]:
def show_table(headers, rows):
    """Таблица без зависимости от pandas."""
    head = ''.join(f'<th style="padding:7px 10px;text-align:right;color:{COLORS["blue"]}">{escape(str(x))}</th>' for x in headers)
    body = ''.join('<tr>'+''.join(f'<td style="padding:6px 10px;text-align:right;border-top:1px solid {COLORS["rule"]}">{escape(str(x))}</td>' for x in row)+'</tr>' for row in rows)
    display(HTML(f'<table style="font-family:{FONT},serif;background:{COLORS["paper"]};color:{COLORS["ink"]};border-collapse:collapse"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))


def show_maneuvers(initial, nu1, solutions, labels=None):
    labels = labels or [chr(65+k) for k in range(len(solutions))]
    rows = []
    for label, s in zip(labels, solutions):
        if s is None:
            continue
        o = s.point
        rows.append([label, f'{o.a:.4f}', f'{o.e:.8f}', f'{o.w/DEG:.5f}',
                     f'{o.i/DEG:.5f}', f'{normalize_angle(o.raan)/DEG:.5f}',
                     f'{s.point.nu/DEG:.5f}', f'{(1000*np.linalg.norm(s.dv)):.4f}', f'{o.rp-R_EARTH:.2f}'])
    show_table(['Решение', 'a₂, км', 'e₂', 'ω₂, °', 'i₂, °', 'Ω₂, °', 'ν₂, °', 'Δv, м/с', 'hп, км'], rows)
    r1, v1 = state_at(initial, nu1)
    print('r₁ [км]   =', r1)
    print('v₁ [м/с]  =', v1*1000)
    for label, s in zip(labels, solutions):
        if s is not None:
            r2, v2 = state_at(s.point)
            print(f'{label}: v₂ [м/с] =', v2*1000)
            print(f'{label}: Δv [м/с] =', s.dv*1000,
                  f'; невязка r = {1000*np.linalg.norm(r2-r1):.3g} м')


def orbit_points(orbit, count=721):
    nu = np.linspace(0, TAU, count)
    P, Q, _ = orbital_basis(orbit)
    return radius_at(orbit, nu)[:, None]*(np.cos(nu)[:, None]*P + np.sin(nu)[:, None]*Q)


def plot_orbits(initial, solutions, nu1=None, spatial=False, title='Орбиты в общей точке'):
    """Реальные эллипсы с общим физическим масштабом; координаты в 10³ км."""
    valid = [(chr(65+k), s) for k, s in enumerate(solutions) if s is not None]
    orbits = [initial] + [s.point for _, s in valid]
    labels = ['Исходная'] + [label for label, _ in valid]
    colors = [COLORS['ink']] + BRANCH_COLORS[:len(valid)]
    styles = ['-'] + BRANCH_STYLES[:len(valid)]
    paths = [orbit_points(o)/1000 for o in orbits]
    fig = plt.figure(figsize=(9.0, 6.0), layout='constrained')
    if spatial:
        ax = fig.add_subplot(111, projection='3d')
        longitude, latitude = np.meshgrid(np.linspace(0,TAU,45), np.linspace(-np.pi/2,np.pi/2,25))
        earth = R_EARTH/1000
        ax.plot_surface(earth*np.cos(latitude)*np.cos(longitude),
                        earth*np.cos(latitude)*np.sin(longitude), earth*np.sin(latitude),
                        color=COLORS['tint'], edgecolor='none', alpha=0.85, shade=True)
        for path, label, color, style in zip(paths, labels, colors, styles):
            ax.plot(*path.T, color=color, ls=style, label=label)
        cloud = np.vstack(paths)
        center = (cloud.max(0)+cloud.min(0))/2
        half = max(np.ptp(cloud, axis=0))/2*1.05
        for setter, coordinate in zip([ax.set_xlim,ax.set_ylim,ax.set_zlim], center):
            setter(coordinate-half, coordinate+half)
        ax.set_box_aspect((1,1,1))
        ax.set_proj_type('ortho')
        ax.view_init(elev=27, azim=-55)
        ax.set(xlabel='X, тыс. км', ylabel='Y, тыс. км', zlabel='Z, тыс. км')
        if nu1 is not None:
            point = state_at(initial, nu1)[0]/1000
            ax.scatter(*point, color=COLORS['ink'], s=25)
            ax.text(*point, '  P')
    else:
        ax = fig.add_subplot(111)
        n, m, _ = node_basis(initial.i, initial.raan)
        project = np.array([n,m])
        for path, label, color, style in zip(paths, labels, colors, styles):
            xy = path@project.T
            ax.plot(*xy.T, color=color, ls=style, label=label)
        ax.add_patch(Circle((0,0), R_EARTH/1000, facecolor=COLORS['tint'], edgecolor=COLORS['rule'], zorder=0))
        ax.set_aspect('equal', adjustable='box')
        ax.set(xlabel='По направлению узла, тыс. км', ylabel='В плоскости орбиты, тыс. км')
        if nu1 is not None:
            point = project@state_at(initial,nu1)[0]/1000
            ax.scatter(*point, color=COLORS['ink'], s=28, zorder=5)
            ax.annotate('P',point,xytext=(6,6),textcoords='offset points')
        ax.grid(True)
    ax.set_title(title)
    ax.legend(loc='upper right')
    return fig


def scan_branches(solver, names=('A','B'), step_deg=0.5):
    """solver(ν₁) -> список Maneuver/None с постоянными номерами ветвей.
    Возвращает сетку [0,360°] (последняя точка повторяет 0), стоимости
    всех математических решений и маску орбит вне Земли. Разрывы — NaN."""
    if step_deg <= 0 or step_deg > 180:
        raise ValueError('Нужен положительный шаг не более 180°.')
    count = int(np.ceil(360/step_deg))
    nus = np.linspace(0,TAU,count+1)
    costs = np.full((len(nus),len(names)), np.nan)
    safe = np.zeros_like(costs,dtype=bool)
    for k, nu in enumerate(nus):
        solutions = solver(normalize_angle(nu))
        if len(solutions) != len(names):
            raise ValueError('Число слотов ветвей должно оставаться постоянным.')
        for j, s in enumerate(solutions):
            if s is not None:
                costs[k,j] = (1000*np.linalg.norm(s.dv))
                safe[k,j] = (s.point.rp > R_EARTH)
    return nus, costs, safe


def plot_scan(nus, costs, safe, names=('A','B'), title='Выбор места манёвра', ymax=None):
    fig, ax = plt.subplots(figsize=(9.5,5.2),layout='constrained')
    for j, name in enumerate(names):
        good = np.where(safe[:,j],costs[:,j],np.nan)
        ax.plot(nus/DEG,good,color=BRANCH_COLORS[j],ls=BRANCH_STYLES[j],label=name)
        bad = np.where(~safe[:,j],costs[:,j],np.nan)
        if np.isfinite(bad).any():
            ax.plot(nus/DEG,bad,color=BRANCH_COLORS[j],ls=':',alpha=.5,
                    label=f'{name}: перицентр внутри Земли')
    ax.set(xlim=(0,360),ylim=(0,ymax),xlabel=r'$\nu_1$, град',ylabel=r'$\|\Delta\mathbf{v}\|$, м/с',title=title)
    ax.set_xticks(np.arange(0,361,60)); ax.grid(True)
    # Легенда вынесена из поля графика, чтобы не закрывать ветви.
    ax.legend(ncol=2,loc='upper center',bbox_to_anchor=(0.5,-0.16))
    return fig


def golden_minimum(function, left, right, iterations=65):
    """Локальное уточнение на уже найденном перебором гладком участке."""
    ratio = (np.sqrt(5)-1)/2
    x1, x2 = right-ratio*(right-left), left+ratio*(right-left)
    f1, f2 = function(x1), function(x2)
    for _ in range(iterations):
        if f1 <= f2:
            right, x2, f2 = x2, x1, f1
            x1 = right-ratio*(right-left); f1 = function(x1)
        else:
            left, x1, f1 = x1, x2, f2
            x2 = left+ratio*(right-left); f2 = function(x2)
    point = (left+right)/2
    return normalize_angle(point), function(point)


def sampled_minima(solver, scan, names=('A','B')):
    """Все обнаруженные сеткой локальные минимумы допустимых ветвей.
    Периодический стык 0/360° учитывается. Сетка не доказывает глобальность."""
    nus, costs, safe = scan
    values = np.where(safe[:-1], costs[:-1], np.inf)
    values = np.where(np.isfinite(values),values,np.inf)
    step = nus[1]-nus[0]
    result = []
    for j, name in enumerate(names):
        for k in range(len(values)):
            value = values[k,j]
            if not np.isfinite(value) or value > values[(k-1)%len(values),j] or value > values[(k+1)%len(values),j]:
                continue
            def objective(nu):
                s = solver(normalize_angle(nu))[j]
                return (1000*np.linalg.norm(s.dv)) if s is not None and (s.point.rp > R_EARTH) else np.inf
            refined = golden_minimum(objective,nus[k]-step,nus[k]+step)
            nu, dv = min([(nus[k],value),refined],key=lambda pair:pair[1])
            if any(row[0]==name and abs(np.arctan2(np.sin(row[1]-nu),np.cos(row[1]-nu)))<1e-5 for row in result):
                continue
            result.append((name,nu,dv,solver(nu)[j].point.rp))
    return sorted(result,key=lambda row:row[2])


def show_minima(minima):
    show_table(['Ветвь','ν₁, °','Δv, м/с','rп,₂, км'],
               [[name,f'{nu/DEG:.5f}',f'{dv:.4f}',f'{rp:.3f}'] for name,nu,dv,rp in minima])


## 2. Компланарный переход между заданными орбитами

Дано: $a_1=10000$ км, $e_1=0{,}1$, $\omega_1=20^\circ$;
$a_2=12000$ км, $e_2=0{,}35$, $\omega_2=90^\circ$.
Плоскость общая, движение в одном направлении. Для этого плоского примера
$\omega$ означает ориентацию перицентра от общей оси $X$.

В общей точке совпадают направление и длина радиус-вектора:
$$\nu_2=\nu_1-(\omega_2-\omega_1),\qquad
\frac{p_1}{1+e_1\cos\nu_1}=\frac{p_2}{1+e_2\cos\nu_2}.$$
Обозначим $\Delta\omega=\omega_2-\omega_1$. После раскрытия косинуса:
$$A\cos\nu_1+B\sin\nu_1+C=0,$$
$$A=e_2p_1\cos\Delta\omega-e_1p_2,\quad
B=e_2p_1\sin\Delta\omega,\quad C=p_1-p_2.$$


In [ ]:
coplanar_initial = Orbit(10000,0.1,20*DEG)
coplanar_target = Orbit(12000,0.35,90*DEG)
delta_w = coplanar_target.w-coplanar_initial.w
A = coplanar_target.e*coplanar_initial.p*np.cos(delta_w)-coplanar_initial.e*coplanar_target.p
B = coplanar_target.e*coplanar_initial.p*np.sin(delta_w)
C = coplanar_initial.p-coplanar_target.p
print(f'p₁={coplanar_initial.p:g} км; p₂={coplanar_target.p:g} км')
print(f'A={A:.7f}, B={B:.7f}, C={C:.7f} км')
coplanar_roots = sorted(solve_acos_bsin_c_eq_0(A,B,C))
coplanar_solutions = []
rows = []
for nu1 in coplanar_roots:
    nu2 = normalize_angle(nu1-delta_w)
    solution = make_maneuver(coplanar_initial,nu1,coplanar_target,nu2)
    coplanar_solutions.append(solution)
    vr1,vt1 = radial_transverse_speed(coplanar_initial,nu1)
    vr2,vt2 = radial_transverse_speed(coplanar_target,nu2)
    rows.append([f'{nu1/DEG:.6f}',f'{nu2/DEG:.6f}',f'{radius_at(coplanar_initial,nu1):.4f}',
                 f'{vr1*1000:.4f}',f'{vt1*1000:.4f}',f'{vr2*1000:.4f}',f'{vt2*1000:.4f}',f'{(1000*np.linalg.norm(solution.dv)):.4f}'])
show_table(['ν₁, °','ν₂, °','r, км','vᵣ₁, м/с','vₜ₁, м/с','vᵣ₂, м/с','vₜ₂, м/с','Δv, м/с'],rows)
for k,(nu1,s) in enumerate(zip(coplanar_roots,coplanar_solutions),1):
    print(f'Точка {k}: Δv [м/с] =', s.dv*1000)
    r1, _ = state_at(coplanar_initial, nu1)
    r2, _ = state_at(s.point)
    print(f'Невязка общей точки: {np.linalg.norm(r2-r1)*1000:.3g} м')
print(f'Первая точка дешевле на {(1000*np.linalg.norm(coplanar_solutions[1].dv))-(1000*np.linalg.norm(coplanar_solutions[0].dv)):.4f} м/с.')
plot_orbits(coplanar_initial,[coplanar_solutions[0]],coplanar_roots[0],title='Два заданных эллипса: первая точка перехода')
plt.show()


Получаем две точки: $\nu_1\approx8{,}823797^\circ$ и $166{,}529685^\circ$.
Импульсы равны примерно $1995{,}9213$ и $1999{,}6815$ м/с.
В общей точке орбит радиальные и трансверсальные направления совпадают.
Компоненты импульса равны $\Delta v_r=v_{r2}-v_{r1}$ и
$\Delta v_t=v_{t2}-v_{t1}$. Модуль импульса:
$$\Delta v=\sqrt{(v_{r2}-v_{r1})^2+(v_{t2}-v_{t1})^2}.$$


## 3. Некомпланарный переход между заданными орбитами

Нормали к орбитам:
$$\widehat{\mathbf h}_j=(\sin i_j\sin\Omega_j,-\sin i_j\cos\Omega_j,\cos i_j)^{\mathsf T}.$$
Направление линии пересечения плоскостей:
$$\mathbf L=\frac{\widehat{\mathbf h}_1\times\widehat{\mathbf h}_2}
{\|\widehat{\mathbf h}_1\times\widehat{\mathbf h}_2\|}.$$
Проверяем **оба** луча $+\mathbf L$ и $-\mathbf L$. Для каждого находим
ориентированные углы от $\mathbf P_j$ до луча и сравниваем радиусы.
Пересечение плоскостей само по себе не означает пересечения орбит.


In [ ]:
spatial_initial = Orbit(18654.3640,0.28969592,265.55428*DEG,35.62718*DEG,89.44357*DEG)
spatial_target = Orbit(20679.0085,0.16932267,234.60634*DEG,38.46693*DEG,107.25881*DEG)
P1,Q1,h1 = orbital_basis(spatial_initial)
P2,Q2,h2 = orbital_basis(spatial_target)
L = normalize(np.cross(h1,h2))
print('ĥ₁ =',h1,'; ĥ₂ =',h2)
print('L =',L)
spatial_angles = []
rows = []
for sign in (1,-1):
    nu1 = directed_angle(P1,sign*L,h1)
    nu2 = directed_angle(P2,sign*L,h2)
    r1,v1 = state_at(spatial_initial,nu1)
    r2,v2 = state_at(spatial_target,nu2)
    residual = np.linalg.norm(r2-r1)
    spatial_angles.append((nu1,nu2))
    rows.append(['+L' if sign==1 else '−L',f'{nu1/DEG:.6f}',f'{nu2/DEG:.6f}',
                 f'{np.linalg.norm(r1):.6f}',f'{np.linalg.norm(r2):.6f}',f'{residual*1000:.3f}'])
show_table(['Луч','ν₁, °','ν₂, °','r₁, км','r₂, км','Невязка, м'],rows)


На $+\mathbf L$ радиусы совпадают с точностью до **0,8 м** из-за округления
исходных элементов в примере. В точной постановке требуется полное совпадение.
Только для воспроизведения этого примера ниже допускаем невязку до 1 м.
На $-\mathbf L$ расхождение около **3992 км**, поэтому переход там невозможен.


In [ ]:
spatial_nu1,spatial_nu2 = spatial_angles[0]
spatial_solution = make_maneuver(spatial_initial,spatial_nu1,spatial_target,spatial_nu2,
                                 position_tolerance_km=0.001)
show_maneuvers(spatial_initial,spatial_nu1,[spatial_solution])
np.testing.assert_allclose((1000*np.linalg.norm(spatial_solution.dv)),789.1805,atol=0.001)
plot_orbits(spatial_initial,[spatial_solution],spatial_nu1,spatial=True,title='Переход между двумя наклонёнными орбитами')
plt.show()


## 4. Коррекция большой полуоси и эксцентриситета

Начальные элементы: $a_1=26000$ км, $e_1=0{,}6$, $\omega_1=250^\circ$,
$i_1=64^\circ$, $\Omega_1=0^\circ$, $\nu_1=60^\circ$.
Цель: $a_2=26554$ км, $e_2=0{,}7$. Сохраняем плоскость и направление обращения.

Сначала вычисляем $r=12800$ км и $u=\omega_1+\nu_1=310^\circ$.
Для конечной орбиты
$$\cos\nu_2=\frac{a_2(1-e_2^2)/r-1}{e_2}=c.$$
Если $|c|<1$, существуют два решения:
$$\nu_{2A}=\arccos c,\quad \nu_{2B}=2\pi-\arccos c,\qquad
\omega_{2A}=u-\nu_{2A},\quad\omega_{2B}=u-\nu_{2B}.$$
При $|c|=1$ решения совпадают, при $|c|>1$ решений нет.
Это то же ограничение, что $a_2(1-e_2)\le r\le a_2(1+e_2)$.


In [ ]:
ae_initial = Orbit(26000,0.6,250*DEG,64*DEG,0)
ae_nu1 = 60*DEG
ae_a2,ae_e2 = 26554,0.7
ae_radius = radius_at(ae_initial,ae_nu1)
ae_cosine = (ae_a2*(1-ae_e2**2)/ae_radius-1)/ae_e2
print(f'r={ae_radius:.4f} км; u={normalize_angle(ae_initial.w+ae_nu1)/DEG:.4f}°; cos ν₂={ae_cosine:.9f}')
ae_solutions = correct_a_e(ae_initial,ae_nu1,ae_a2,ae_e2)
show_maneuvers(ae_initial,ae_nu1,ae_solutions)
plot_orbits(ae_initial,ae_solutions,ae_nu1,title='Коррекция a,e: два положения перицентра')
plt.show()


Получаем $\nu_{2A}=85{,}2463^\circ$, $\nu_{2B}=274{,}7537^\circ$.
Аргументы перицентра равны $224{,}7537^\circ$ и $35{,}2463^\circ$.
Затраты составляют примерно $1388{,}8382$ и $6358{,}3266$ м/с.

### Выбор места манёвра

Теперь сохраняем исходную орбиту и целевые $a_2,e_2$, но меняем $\nu_1$.
В каждой точке заново определяем конечные элементы и импульс.
Перебираем полный оборот с шагом $0{,}5^\circ$, затем уточняем найденные минимумы.


In [ ]:
ae_solver = lambda nu: correct_a_e(ae_initial,nu,ae_a2,ae_e2)
ae_scan = scan_branches(ae_solver)
plot_scan(*ae_scan,title='Коррекция a,e: выбор места импульса')
plt.show()
ae_minima = sampled_minima(ae_solver,ae_scan)
show_minima(ae_minima)


Две симметричные оптимальные точки находятся вблизи $145{,}9067^\circ$ и
$214{,}0933^\circ$. Минимальные затраты — около $461{,}0615$ м/с.
При поиске сравниваем **все** ветви; заранее выбирать A или B нельзя.


## 5. Коррекция большой полуоси и аргумента перицентра

Исходные элементы: $a_1=26222$ км, $e_1=0{,}7$, $\omega_1=266^\circ$,
$i_1=63{,}434^\circ$, $\Omega_1=0^\circ$, $\nu_1=222^\circ$.
Цель: $a_2=26554$ км, $\omega_2=270^\circ$.

Плоскость и направление обращения сохраняются. Поэтому
$$u=\omega_1+\nu_1,\qquad \nu_2=u-\omega_2\pmod{2\pi}.$$
Из $r=a_2(1-e_2^2)/(1+e_2\cos\nu_2)$ получаем квадратное уравнение
$$a_2e_2^2+r\cos\nu_2\,e_2+(r-a_2)=0.$$
Сначала находим **все** действительные корни, затем оставляем $0<e_2<1$.
$e_2=0$ не подходит для постановки с заданным перицентром, а $e_2\ge1$
не задаёт требуемый эллипс. После этого проверяем радиус перицентра.


In [ ]:
aw_initial = Orbit(26222,0.7,266*DEG,63.434*DEG,0)
aw_nu1, aw_a2, aw_w2 = 222*DEG,26554,270*DEG
aw_radius = radius_at(aw_initial,aw_nu1)
aw_nu2 = normalize_angle(aw_initial.w+aw_nu1-aw_w2)
print(f'r={aw_radius:.6f} км; ν₂={aw_nu2/DEG:.6f}°')
print('Коэффициенты квадратного уравнения:',aw_a2,aw_radius*np.cos(aw_nu2),aw_radius-aw_a2)
print('Корни e₂:',solve_quadratic(aw_a2,aw_radius*np.cos(aw_nu2),aw_radius-aw_a2))
aw_solutions = correct_a_w(aw_initial,aw_nu1,aw_a2,aw_w2)
show_maneuvers(aw_initial,aw_nu1,aw_solutions)
plot_orbits(aw_initial,aw_solutions,aw_nu1,title='Коррекция a,ω: два математических эллипса')
plt.show()


Решение A: $e_2\approx0{,}76197244$, $\Delta v\approx338{,}6026$ м/с,
но $r_{\text{п},2}\approx6320{,}6$ км **меньше радиуса Земли**.
Решение B: $e_2\approx0{,}06516787$, $\Delta v\approx2626{,}4796$ м/с;
эта орбита не пересекает Землю.
Поэтому меньший математический импульс не всегда даёт допустимую орбиту.

### Выбор места манёвра и несколько локальных минимумов

Сохраняем имена ветвей: A — больший корень квадратного уравнения,
B — меньший. После удаления недопустимого корня нельзя переименовывать
оставшуюся ветвь. График построен в диапазоне до 3000 м/с, как в лекции.
Математические решения с перицентром внутри Земли показаны светлым пунктиром.


In [ ]:
aw_solver = lambda nu: correct_a_w(aw_initial,nu,aw_a2,aw_w2)
aw_scan = scan_branches(aw_solver)
plot_scan(*aw_scan,title='Коррекция a,ω: выбор места импульса',ymax=3000)
plt.show()
aw_minima = sampled_minima(aw_solver,aw_scan)
show_minima(aw_minima)


Меньшие затраты среди обнаруженных допустимых минимумов составляют примерно
$197{,}4172$ м/с при $\nu_1=115{,}7265^\circ$ (ветвь A).
Есть и другие локальные минимумы, в том числе около $179{,}5419^\circ$
и $264{,}5192^\circ$. Один запуск локального оптимизатора не заменяет
перебор всего оборота и сравнение ветвей.


## 6. Коррекция большой полуоси, эксцентриситета и ДВУ

Берём исходное состояние из примера $a,e$, но дополнительно задаём
$\Omega_2=3^\circ$.
Новая плоскость должна содержать неизменный радиус-вектор:
$$r_x\sin i_2\sin\Omega_2-r_y\sin i_2\cos\Omega_2+r_z\cos i_2=0.$$
При заданной ДВУ находим $i_2$ в диапазоне $0<i_2<\pi$:
$$\tan i_2=\frac{-r_z}{r_x\sin\Omega_2-r_y\cos\Omega_2}.$$
В коде используем `atan2`, чтобы не терять четверть при нулевом или отрицательном
знаменателе. Экваториальные и вырожденные случаи рассматриваются отдельно.

После поворота плоскости **пересчитываем** аргумент широты:
$$\widehat{\mathbf n}_2=(\cos\Omega_2,\sin\Omega_2,0),\quad
\widehat{\mathbf m}_2=\widehat{\mathbf h}_2\times\widehat{\mathbf n}_2,$$
$$u_2=\operatorname{atan2}(\mathbf r\cdot\widehat{\mathbf m}_2,
\mathbf r\cdot\widehat{\mathbf n}_2)\pmod{2\pi}.$$
Эквивалентно, $\cos u_2=(r_x\cos\Omega_2+r_y\sin\Omega_2)/r$,
$\sin u_2=r_z/(r\sin i_2)$.
Затем находим прежние два решения для $\nu_2$ и для каждого $\omega_2=u_2-\nu_2$.


In [ ]:
node_O2 = 3*DEG
node_r1,_ = state_at(ae_initial,ae_nu1)
node_i2 = plane_with_raan(node_r1,node_O2)
node_u2 = argument_latitude(node_r1,node_i2,node_O2)
print(f'i₂={node_i2/DEG:.8f}°; cos u₂={np.cos(node_u2):.9f}; sin u₂={np.sin(node_u2):.9f}; u₂={node_u2/DEG:.8f}°')
node_solutions = correct_a_e_raan(ae_initial,ae_nu1,ae_a2,ae_e2,node_O2)
show_maneuvers(ae_initial,ae_nu1,node_solutions)
plot_orbits(ae_initial,node_solutions,ae_nu1,spatial=True,title='Коррекция a,e,Ω: две орбиты в новой плоскости')
plt.show()


Получаем $i_2\approx61{,}812067^\circ$, $u_2\approx308{,}633150^\circ$.
Аргументы перицентра: $223{,}386862^\circ$ и $33{,}879438^\circ$.
Импульсы: $1435{,}7652$ и $6368{,}7414$ м/с.

### Выбор места пространственного манёвра

В каждой точке заново находим $i_2,u_2,\nu_2,\omega_2$.
Сохранение старого $u_1$ после изменения плоскости дало бы ошибочное положение аппарата.
В точках, где заданная ДВУ возможна только для экваториальной орбиты,
узел не определён: такие точки исключаются, а линии графика разрываются.


In [ ]:
node_solver = lambda nu: correct_a_e_raan(ae_initial,nu,ae_a2,ae_e2,node_O2)
node_scan = scan_branches(node_solver)
plot_scan(*node_scan,title='Коррекция a,e,Ω: выбор места импульса')
plt.show()
node_minima = sampled_minima(node_solver,node_scan)
show_minima(node_minima)


Наименьший найденный допустимый импульс — около $474{,}4646$ м/с
при $\nu_1\approx213{,}1272^\circ$ (ветвь B).
На ветви A есть минимум около $150{,}1500^\circ$ с затратами $494{,}1414$ м/с.

### Что меняется, если вместо ДВУ задано наклонение

То же условие плоскости при заданном $i_2$ становится тригонометрическим
уравнением относительно $\Omega_2$:
$$r_x\sin i_2\sin\Omega_2-r_y\sin i_2\cos\Omega_2+r_z\cos i_2=0.$$
Могут существовать две различные плоскости. В каждой из них заданные
$a_2,e_2$ могут допускать две истинные аномалии. Так возникает до четырёх
конечных орбит. Эту постановку исследуем в необязательном задании.


# Домашнее задание 2

Ниже находятся заготовки для **ваших** решений. Готовые примеры выше менять не требуется.
Для каждого задания приведите вывод нужных формул, расчёты, таблицы/графики и выводы.
Можно добавлять кодовые и текстовые ячейки.

Общая исходная орбита:
$$a_1=26000\text{ км},\quad e_1=0{,}6,\quad\omega_1=250^\circ,
\quad i_1=64^\circ,\quad\Omega_1=0^\circ.$$
Сначала выполняем манёвр при $\nu_1=60^\circ$.
Рассматриваем только некруговые эллипсы $0<e_2<1$, $r_{\text{п},2}>R_\oplus$.
Пространственные варианты с $i_2>90^\circ$ тоже учитываем.
Все конечные орбиты должны проходить через точку импульса.


In [ ]:
hw_initial = Orbit(a=26000,e=0.6,w=250*DEG,i=64*DEG,raan=0)
hw_nu1 = 60*DEG
hw_r1,hw_v1 = state_at(hw_initial,hw_nu1)


## Задание 1. Коррекция радиуса перицентра и аргумента перицентра

Требуется получить $r_{\text{п},2}=10000$ км, $\omega_2=245^\circ$.
Плоскость и направление обращения сохраняются.

1. Из $r_{\text{п},2}=a_2(1-e_2)$ и уравнения орбиты получите выражения для
   $e_2,a_2$ в известной точке. Сначала найдите $u$ и $\nu_2$.
   Сформулируйте условия существования допустимого решения.
2. При $\nu_1=60^\circ$ найдите конечные элементы, вектор импульса в неподвижных
   осях и его модуль. Проверьте общую точку и радиус перицентра.
3. Переберите $0\le\nu_1<360^\circ$ с шагом не более $0{,}5^\circ$.
   Постройте $\Delta v(\nu_1)$ с разрывами там, где допустимого эллипса нет.
4. Найдите наименьшие затраты на сетке, при желании уточните минимум.
   Сравните с манёвром при $60^\circ$. Объясните, почему этот набор элементов
   допускает независимую коррекцию не в любой точке.

**Вывод формул и условия существования:**

*Впишите решение.*


In [ ]:
hw1_rp2 = 10000.0  # км
hw1_w2 = 245*DEG

# Ваш код: конечная орбита, make_maneuver, таблица, проверка общей точки.


In [ ]:
# Ваш код: функция решения при произвольной ν₁, перебор и график.
# Для одной ветви: names=('A',).


**Результаты, минимум и объяснение ограничений:**

*Впишите ответ.*


## Задание 2. Коррекция большой полуоси, эксцентриситета и аргумента перицентра

При $\nu_1=60^\circ$ требуется получить
$a_2=28000$ км, $e_2=0{,}65$, $\omega_2=240^\circ$.
Наклонение и ДВУ заранее не задаются. Найдите **все** допустимые варианты,
включая обратные орбиты.

1. Найдите возможные $\nu_2$ из уравнения радиуса. Для каждой определите
   $u_2=\omega_2+\nu_2$ и найдите допустимые наклонения из
   $r_z=r\sin i_2\sin u_2$. Объясните, почему некоторых ветвей может не быть.
2. Для каждого $i_2$ восстановите $\Omega_2$ по первым двум координатам
   радиус-вектора. Используйте обе координаты и `atan2`, чтобы выбрать четверть.
3. Найдите все конечные элементы и импульсы. Проверьте $\mathbf r_2=\mathbf r_1$,
   заданные $a_2,e_2,\omega_2$ и $r_{\text{п},2}>R_\oplus$.
   Составьте таблицу элементов и затрат при $\nu_1=60^\circ$.
4. Переберите $0\le\nu_1<360^\circ$ с шагом не более $0{,}5^\circ$.
   Постройте на одном графике $\Delta v(\nu_1)$ для прямой и обратной ветвей,
   добавьте легенду. Недопустимые точки пропускайте, не соединяя их линией.
   Найдите наиболее экономичный манёвр и сравните с результатом при $60^\circ$.
5. Покажите исходную и найденные орбиты на одном пространственном рисунке.
   Объясните, почему коррекция $\omega$ здесь возможна вместе с заданными $a,e$,
   хотя для плоского импульса эти три изменения обычно зависимы.

**Вывод формул и разбор ветвей:**

*Впишите решение.*


In [ ]:
hw2_a2 = 28000.0  # км
hw2_e2 = 0.65
hw2_w2 = 240*DEG

# Ваш код: допустимые ν₂, затем i₂ и Ω₂, список конечных орбит.


In [ ]:
# Ваш код: make_maneuver, таблица при ν₁=60°, проверки, plot_orbits(..., spatial=True).
# Перебор ν₁: график Δv(ν₁) для прямой и обратной ветвей, поиск минимума.


**Число решений, сравнение затрат и объяснение:**

*Впишите ответ.*


## Задание 3*. Коррекция большой полуоси, эксцентриситета и наклонения (необязательное)

Требуется получить $a_2=28000$ км, $e_2=0{,}65$, $i_2=70^\circ$.
ДВУ и аргумент перицентра определяются из решения.

1. При $\nu_1=60^\circ$ решите условие
   $\widehat{\mathbf h}_2\cdot\mathbf r=0$ относительно $\Omega_2$.
   В каждой найденной плоскости найдите обе конечные орбиты с заданными $a_2,e_2$.
   Получите четыре различных решения, проверьте общую точку и целевые элементы.
2. Обозначьте решения $1A,1B,2A,2B$: номер соответствует одной из двух плоскостей,
   буква — одной из двух конечных истинных аномалий.
   Покажите таблицу элементов и импульсов при $60^\circ$.
3. Переберите полный оборот с шагом не более $0{,}5^\circ$.
   Постройте **четыре линии на одном графике** $\Delta v(\nu_1)$.
   Сохраняйте происхождение ветвей при переходе углов через $0/360^\circ$:
   не переупорядочивайте их по текущему значению ДВУ или величине импульса.
4. Найдите лучший результат для каждой ветви и среди всех четырёх.
5. Объясните происхождение четырёх решений. При каких условиях две плоскости
   или две орбиты внутри плоскости сливаются? Почему для произвольных исходных
   и целевых данных число решений может быть меньше четырёх?

**Вывод уравнения для ДВУ и схема объединения ветвей:**

*Впишите решение.*


In [ ]:
hw3_a2 = 28000.0  # км
hw3_e2 = 0.65
hw3_i2 = 70*DEG
hw3_names = ('1A','1B','2A','2B')

# Ваш код: две плоскости из тригонометрического уравнения.
# Для каждой используйте correct_a_e(..., plane=(i2, raan2)).


In [ ]:
# Ваш код: таблица четырёх решений при ν₁=60°, проверки.


In [ ]:
# Ваш код: scan_branches для четырёх решений, plot_scan и поиск минимумов.


**Сравнение ветвей, минимумов и условия слияния решений:**

*Впишите ответ.*

## Что сдать

Сдайте **этот ноутбук с решёнными заданиями 1 и 2** и, по желанию, заданием 3*.
Все вспомогательные функции уже написаны выше: для решения нужно корректно
объединить их вызовы, задать условия отбора решений и пояснить расчёты.
Формулы и выводы оформите в текстовых ячейках, вычисления и графики — в кодовых.

Перед отправкой перезапустите ядро, выполните все ячейки сверху вниз и сохраните
ноутбук с результатами и графиками. Назовите файл
**`hw_2_name_surname.ipynb`**, заменив `name_surname` своими именем и фамилией латиницей.
